In [1]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [3]:
import sys

sys.path.append('../../scripts')

In [4]:
import numpy as np
import scanpy as sc
import os
DATA_ROOT = '/data/a330d' #os.environ.get("DATA_ROOT", ".")
import matplotlib.pyplot as plt
import decoupler as dc
import scipy.sparse as sp
import pandas as pd
from scgraph import scGraph

from scipy.stats import pearsonr, spearmanr

from cellina import make_neighbor_perturbation
from cellina_graph import make_perturbed_expression
from utils import set_seed
from train_loo import preprocess_crc, preprocess_merfish, _load_model, split_indices, preprocess_spatial_features
from counterfactual_analysis import compute_rmse, compute_edistance, mixing_index, get_lfc, precision, direction_match, compute_mse_lfc, _to_dense
from counterfactual_analysis import get_perturbation_logfc, get_global_perturbation_logfc
from configs.adata_crc_config import ADATA_ARGS as ADATA_ARGS_CRC
from configs.adata_merfish_config import ADATA_ARGS as ADATA_ARGS_MERFISH

In [5]:
import cellina

cellina.__version__

'0.7.4'

In [6]:
set_seed(0)

In [7]:
DATASET_NAME = "merfish"  # or "merfish"
CELLINA_BASE_MODEL_ROOT = os.path.join(DATA_ROOT, "data/ood/trained")
CELLINA_GAT_MODEL_ROOT = os.path.join(DATA_ROOT, "data/ood/trained")

In [17]:
CRC_PATHS = [
    os.path.join(DATA_ROOT, "datasets/crc/raw_zenodo/crc_120.h5ad"),
    os.path.join(DATA_ROOT, "datasets/crc/raw_zenodo/crc_210.h5ad"),
    os.path.join(DATA_ROOT, "datasets/crc/raw_zenodo/crc_221.h5ad"),
    os.path.join(DATA_ROOT, "datasets/crc/raw_zenodo/crc_231.h5ad"),
    os.path.join(DATA_ROOT, "datasets/crc/raw_zenodo/crc_232.h5ad"),
    os.path.join(DATA_ROOT, "datasets/crc/raw_zenodo/crc_242.h5ad"),
]

CRC_HOLDOUTS = [
    "Endothelial",
    "Epithelial",
    "Fibroblast",
    "Myeloid",
    "T_cell",
]

MERFISH_PATHS = [
    os.path.join(DATA_ROOT, "datasets/MERFISH_mouse_brain/C57BL6J-2.036.h5ad"),    
    #os.path.join(DATA_ROOT, "datasets/MERFISH_mouse_brain/C57BL6J-2.039.h5ad"),
    #os.path.join(DATA_ROOT, "datasets/MERFISH_mouse_brain/C57BL6J-2.041.h5ad"),
]

MERFISH_HOLDOUTS = [
    'glutamatergic neuron',
    'oligodendrocyte',
    'astrocyte',
    'GABAergic neuron',
    'endothelial cell',
]

PATHS = CRC_PATHS if DATASET_NAME == "crc" else MERFISH_PATHS
HOLDOUT_CELLTYPES = CRC_HOLDOUTS if DATASET_NAME == "crc" else MERFISH_HOLDOUTS
DATA_ARGS = ADATA_ARGS_CRC if DATASET_NAME == "crc" else ADATA_ARGS_MERFISH
COUNTS_PER_K = 1e4

In [18]:
n_top_genes = DATA_ARGS.get('n_top_genes')
labels_key = DATA_ARGS.get('labels_key')
domains_key = DATA_ARGS.get('domains_key')
batch_key = DATA_ARGS.get('batch_key')
control_domain = DATA_ARGS.get('control_domains')[0]
holdout_domains = DATA_ARGS.get('holdout_domains')
n_neighbors = DATA_ARGS.get('n_neighbors')
batch_size = 512
library_size = 'latent'
n_deg = 50
n_pert_genes = 200

In [19]:
# Create SLIDES which contain file names from PATHS - first split by "/" and take last part, then split by "." and take first part
SLIDES = [path.split("/")[-1].split(".h5ad")[0] for path in PATHS]

In [23]:
save_path = '/data/a330d/datasets/scgraph'
model_names = ['cellina-W_1'] #['cellina-graph-W']#, 

for path, slide_id in zip(PATHS, SLIDES):
    adata = sc.read(path)
    
    if DATASET_NAME == 'crc':
        adata = preprocess_crc(adata, n_top_genes=n_top_genes, labels_key=labels_key, domains_key=domains_key)
    elif DATASET_NAME == 'merfish':
        adata = preprocess_merfish(adata, n_top_genes=n_top_genes, labels_key=labels_key, domains_key=domains_key)
    else:
        raise ValueError(f"Unknown dataset_name: {DATASET_NAME}. Supported: crc, merfish")
    
    for holdout_celltype in HOLDOUT_CELLTYPES:
        # 50 times * in print
        print(f"{'='*50} Slide: {slide_id}, Holdout Celltype: {holdout_celltype} {'='*50}")
        # create splits
        train_idx, val_idx, test_idx = split_indices(adata,
                                                    holdout_celltype,
                                                    labels_key=labels_key,
                                                    domains_key=domains_key,
                                                    holdout_domains=holdout_domains,
                                                    seed=0)

        splits = (train_idx, val_idx, test_idx)
        # Compute spatial features after splitting to avoid data leakage
        step_size_px = 0.12028 if DATASET_NAME == 'crc' else 0.109
        adata = preprocess_spatial_features(adata, step_size_px=step_size_px, n_neighbors=n_neighbors, test_indices=test_idx)
        
        for model_name in model_names:
            if model_name == 'cellina-W_1':
                 model_class = "cellina"
            else:
                model_class = "cellina_graph"
            
                        
            MODEL_ROOT = CELLINA_BASE_MODEL_ROOT if model_class == 'cellina' else CELLINA_GAT_MODEL_ROOT
            save_dir = os.path.join(MODEL_ROOT, slide_id, holdout_celltype, model_name)
            
            try:
                model = _load_model(save_dir,
                                    model_class=model_class,
                                    adata=adata,
                                    splits=splits)
            except Exception as e:
                print(f"Failed to load model from {save_dir} with error: {e}")
                continue

            # Compute latents and store in adata.obsm
            adata.obsm[f'X_cellina_{holdout_celltype}_z'] = model.get_latent_representation(latent_key='z', batch_size=batch_size)
            adata.obsm[f'X_cellina_{holdout_celltype}_s'] = model.get_latent_representation(latent_key='s', batch_size=batch_size)

    # Drop a list of keys from adata.obsm
    keys_to_drop = ['X_CCF', 'X_spatial_coords', 'spatial_x', 'spatial']
    for key in keys_to_drop:
        adata.obsm.pop(key, None)
    adata.write_h5ad(f"{save_path}/{slide_id}.h5ad")

================================================== Slide: C57BL6J-2.036, Holdout Celltype: glutamatergic neuron ==================================================


Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.


INFO     File /data/a330d/data/ood/trained/C57BL6J-2.036/glutamatergic neuron/cellina-W_1/model.pt already         
         downloaded                                                                                                
INFO     cellina: The Cellina model has been initialized with adversarial domain forgetting                        
cellina loaded model from /data/a330d/data/ood/trained/C57BL6J-2.036/glutamatergic neuron/cellina-W_1
================================================== Slide: C57BL6J-2.036, Holdout Celltype: oligodendrocyte ==================================================


Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.


INFO     File /data/a330d/data/ood/trained/C57BL6J-2.036/oligodendrocyte/cellina-W_1/model.pt already downloaded   
INFO     cellina: The Cellina model has been initialized with adversarial domain forgetting                        
cellina loaded model from /data/a330d/data/ood/trained/C57BL6J-2.036/oligodendrocyte/cellina-W_1
================================================== Slide: C57BL6J-2.036, Holdout Celltype: astrocyte ==================================================


...

INFO     File /data/a330d/data/ood/trained/C57BL6J-2.036/astrocyte/cellina-W_1/model.pt already downloaded         
INFO     cellina: The Cellina model has been initialized with adversarial domain forgetting                        
cellina loaded model from /data/a330d/data/ood/trained/C57BL6J-2.036/astrocyte/cellina-W_1
================================================== Slide: C57BL6J-2.036, Holdout Celltype: GABAergic neuron ==================================================


Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.


INFO     File /data/a330d/data/ood/trained/C57BL6J-2.036/GABAergic neuron/cellina-W_1/model.pt already downloaded  
INFO     cellina: The Cellina model has been initialized with adversarial domain forgetting                        
cellina loaded model from /data/a330d/data/ood/trained/C57BL6J-2.036/GABAergic neuron/cellina-W_1
================================================== Slide: C57BL6J-2.036, Holdout Celltype: endothelial cell ==================================================


Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.


INFO     File /data/a330d/data/ood/trained/C57BL6J-2.036/endothelial cell/cellina-W_1/model.pt already downloaded  
INFO     cellina: The Cellina model has been initialized with adversarial domain forgetting                        
cellina loaded model from /data/a330d/data/ood/trained/C57BL6J-2.036/endothelial cell/cellina-W_1


In [24]:
adata

AnnData object with n_obs × n_vars = 46540 × 1120
    obs: 'donor_id', 'development_stage_ontology_term_id', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'disease_ontology_term_id', 'tissue_ontology_term_id', 'cell_type_ontology_term_id', 'assay_ontology_term_id', 'suspension_type', 'cluster_id_transfer', 'subclass_transfer', 'cluster_confidence_score', 'subclass_confidence_score', 'high_quality_transfer', 'major_brain_region', 'ccf_region_name', 'brain_section_label', 'tissue_type', 'is_primary_data', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid', 'n_counts', 'is_holdout', '_scvi_batch', '_scvi_labels', '_scvi_domains'
    var: 'gene_name', 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type', 'n_counts', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
    uns: 'citation', 'organism', 'organism_onto

In [26]:
f"{save_path}/{slide_id}.h5ad"

'/data/a330d/datasets/scgraph/C57BL6J-2.036.h5ad'

In [ ]:
scgraph = scGraph(
    adata_path=f"{save_path}/{slide_id}.h5ad",   # Path to AnnData object
    batch_key=batch_key,                     # Column name for batch information
    label_key=labels_key,                     # Column name for cell type labels
    trim_rate=0.05,                          # Trim rate for robust mean calculation
    thres_batch=100,                       # Minimum number of cells per batch
    thres_celltype=10,                       # Minimum number of cells per cell type
    only_umap=True,                          # Only evaluate 2D embeddings (mostly umaps)
)

# Run the analysis, return a pandas dataframe
results = scgraph.main()

# Save the results
results.to_csv(f"{save_path}/{slide_id}_embedding_evaluation_results.csv")